In [1]:
%pip install -q python-dotenv openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import dotenv
dotenv.load_dotenv()

True

In [3]:
system_prompt = '''
You are an AI assistant who can perform the following steps:
1. Reason through the problem by describing your thoughts in a "Thought:" section.
2. When you need to use a tool, output an "Action:" section with the tool name and its input.
3. After the tool call, you'll see an "Observation:" section with the tool's output.
4. Continue this cycle of Thought → Action → Observation as needed.
5. End with a concise "Final Answer:" that answers the user's query.

Note:
- The chain of thought in "Thought:" sections is only visible to you and not part of your final answer.
- The user should only see your "Final Answer:".
'''

In [4]:
user_prompt = '''
What is the weather in Nairobi,Kenya Today?
'''

In [5]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt}
    ]
)

In [6]:
text = completion.choices[0].message.content
print(text)

Thought: To provide the current weather information for Nairobi, Kenya, I need to access a weather service or tool that can give real-time weather data for that location.

Action: GetWeather("Nairobi, Kenya") 

(Note: As an AI, I don't have direct access to the internet or live data, so I can't perform this action. However, I can guide the user to check a weather website or app for the current weather in Nairobi.) 

Observation: N/A (Awaiting action output)

Final Answer: Please check a reliable weather website or application for the current weather in Nairobi, Kenya.


In [7]:
import re
pattern = r'Action:\s*(\w+)\("([^"]+)"\)'

match = re.search(pattern, text)
if match:
    tool_name = match.group(1)    # 'GetWeather'
    tool_input = match.group(2)   
    print("Tool name:", tool_name)
    print("Tool input:", tool_input)
else:
    print("No match found.")

Tool name: GetWeather
Tool input: Nairobi, Kenya


In [22]:
import requests
import os

def get_current_weather(city_name):
    
   weather_info = {
    "city": "Nairobi",
    "temperature": 25,       # in Celsius (current)
    "description": "Partly sunny",
    "humidity": 60          # approximate average humidity in percentage
   }
   return weather_info

In [23]:
if tool_name == 'GetWeather':
    weather_info = get_current_weather(tool_input)
    print(weather_info)

{'city': 'Nairobi', 'temperature': 25, 'description': 'Partly sunny', 'humidity': 60}


In [24]:
updated_text = text + f"\n\n Observation: {weather_info}"
print(updated_text)

Thought: To provide the current weather information for Nairobi, Kenya, I need to access a weather service or tool that can give real-time weather data for that location.

Action: GetWeather("Nairobi, Kenya") 

(Note: As an AI, I don't have direct access to the internet or live data, so I can't perform this action. However, I can guide the user to check a weather website or app for the current weather in Nairobi.) 

Observation: N/A (Awaiting action output)

Final Answer: Please check a reliable weather website or application for the current weather in Nairobi, Kenya.

 Observation: {'city': 'Nairobi', 'temperature': 25, 'description': 'Partly sunny', 'humidity': 60}


In [25]:
completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt},
        {"role": "assistant","content": text},
        {"role": "user","content": updated_text}
    ]
)

In [26]:
text2 = completion.choices[0].message.content
print(text2)

Final Answer: The current weather in Nairobi, Kenya is partly sunny with a temperature of 25°C and 60% humidity.
